In [3]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X =  torch.randn(100, 6)
y = torch.randint(0, 3, size=(100,), dtype=torch.long)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=20, shuffle=False)

print(f"number of batches: {len(loader)}")

for batch_x, batch_y in loader:
    print(batch_x.shape, batch_y.shape)

number of batches: 5
torch.Size([20, 6]) torch.Size([20])
torch.Size([20, 6]) torch.Size([20])
torch.Size([20, 6]) torch.Size([20])
torch.Size([20, 6]) torch.Size([20])
torch.Size([20, 6]) torch.Size([20])


In [4]:
X = torch.randn(100, 4)
y_sorted = torch.tensor([0] * 50 + [1] * 50, dtype=torch.long)

dataset = TensorDataset(X, y_sorted)

loader_no_shuffle = DataLoader(dataset, batch_size=10, shuffle=False)
loader_shuffle = DataLoader(dataset, batch_size=10, shuffle=True)

batch_x, batch_y = next(iter(loader_no_shuffle))
print(f"No shuffle first labels:{batch_y}")

batch_x, batch_y = next(iter(loader_shuffle))
print(f"Shuffle first labels:{batch_y}")

No shuffle first labels:tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Shuffle first labels:tensor([0, 0, 1, 0, 0, 1, 0, 0, 1, 0])


In [11]:
import torch
from torch.utils.data import TensorDataset, random_split

X = torch.randn(100, 6)
y = torch.randint(0, 3, size=(100,), dtype=torch.long)

dataset = TensorDataset(X, y)

train_size = 70
valid_size = 15
test_size = 15

train_dataset, valid_dataset, test_dataset = random_split(
    # dataset: Dataset[_T@random_split],
    dataset = dataset,
    # lengths: Sequence[int | float],
    lengths = [train_size, valid_size, test_size]
    # generator: Generator | None = default_generator
)

print(f"train length: {len(train_dataset)}")
print(f"valid length: {len(valid_dataset)}")
print(f"test length: {len(test_dataset)}")


train length: 70
valid length: 15
test length: 15


In [12]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

train_batch_x, train_batch_y = next(iter(train_loader))
valid_batch_x, valid_batch_y = next(iter(valid_loader))
test_batch_x, test_batch_y = next(iter(test_loader))

print(f"X shape: {X.shape}")
print(f"train batch: {train_batch_x.shape},{train_batch_y.shape}")
print(f"valid batch: {valid_batch_x.shape},{valid_batch_y.shape}")
print(f"test batch: {test_batch_x.shape},{test_batch_y.shape}")

X shape: torch.Size([100, 6])
train batch: torch.Size([16, 6]),torch.Size([16])
valid batch: torch.Size([15, 6]),torch.Size([15])
test batch: torch.Size([15, 6]),torch.Size([15])


In [16]:
import torch
from torch.utils.data import Dataset, TensorDataset, random_split

class SubsetWithTransform(Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_dataset[self.indices[idx]]
        if self.transform is not None:
            x = self.transform(x)
        return x, y

X = torch.arange(100 * 6, dtype=torch.float32).reshape(100, 6)
y = torch.arange(100) % 3
raw_dataset = TensorDataset(X, y)

split_generator = torch.Generator().manual_seed(42)
train_part, valid_part, test_part = random_split(
    # dataset: Dataset[_T@random_split],
    dataset = raw_dataset,
    # lengths: Sequence[int | float],
    lengths = [70, 15, 15],
    # generator: Generator | None = default_generator
    generator = split_generator
)

train_transform = lambda x: x + torch.rand_like(x) * 0.01
valid_transform = lambda x: x
test_transform = lambda x: x

train_dataset = SubsetWithTransform(raw_dataset,train_part.indices, train_transform)
valid_dataset = SubsetWithTransform(raw_dataset,valid_part.indices, valid_transform)
test_dataset = SubsetWithTransform(raw_dataset,test_part.indices, test_transform)

print(f"{len(train_dataset), len(valid_dataset), len(test_dataset)}")
print(f"valid sample is stable: {torch.equal(valid_dataset[0][0], valid_dataset[0][0])}")

(70, 15, 15)
valid sample is stable: True


In [ ]:
try:
    random_split(dataset, [70, 20])
except ValueError as e:
    print(e)

# Sum of input lengths does not equal the length of the input dataset!

Sum of input lengths does not equal the length of the input dataset!


In [21]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

# 다음 조건을 만족하는 train/valid/test DataLoader를 만드세요.
# 1. 전체 샘플 수는 120개입니다.
# 2. feature 수는 10개입니다.
# 3. class 수는 4개입니다.
# 4. train/valid/test 크기는 90개, 15개, 15개입니다.
# 5. batch size는 15입니다.
# 6. train loader는 `shuffle=True`, valid/test loader는 `shuffle=False`입니다.
# 7. 세 loader의 첫 batch shape를 출력하세요.

X = torch.randn(120, 10)
y = torch.randint(0, 3, size=(120,),dtype=torch.long)

dataset = TensorDataset(X, y)

generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset, test_dataset = random_split(
    dataset = dataset,
    lengths = [90, 15, 15],
    generator = generator
)

train_loader = DataLoader(train_dataset, batch_size=15, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=15, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=15, shuffle=True)

for name, loader in [
    ("train", train_loader),
    ("valid", valid_loader),
    ("test", test_loader),
]:
    batch_x, batch_y = next(iter(loader))
    print(f"{name}, {batch_x.shape}, {batch_y.shape}")

train, torch.Size([15, 10]), torch.Size([15])
valid, torch.Size([15, 10]), torch.Size([15])
test, torch.Size([15, 10]), torch.Size([15])
